# Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import libraries used in this notebook
import os
import cv2
import numpy as np
import pandas as pd
import random
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.utils as vutils
import torchvision.transforms as transforms
from torchvision.io import read_image
from torch.utils.data import Dataset, DataLoader

# Set File path for Image Data
#data_filepath = '/content/drive/MyDrive/data/chest_xray'
data_filepath = '../data/chest_xray'

In [ ]:
import torch

# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

# Dataset Loading

In [ ]:
# Create a DataFrame of image paths and their corresponding numeric label (0 = Normal, 1 = Pneumonia)
image_dirs = ['train']  # ✅ Only using training dataset for now
data = []

# ✅ Define label mapping: 'NORMAL' → 0, 'PNEUMONIA' → 1
label_mapping = {"NORMAL": 0, "PNEUMONIA": 1}

# Loop through Image Directories to find images
for dir in image_dirs:
    dir_path = os.path.join(data_filepath, dir)
    
    for label_name in ["NORMAL", "PNEUMONIA"]:
        label_dir = os.path.join(dir_path, label_name)
        
        for file_name in os.listdir(label_dir):
            file_path = os.path.join(label_dir, file_name)

            if file_name.endswith(('.png', '.jpg', '.jpeg')):  # ✅ Ensure valid image file
                data.append([dir, file_path, label_mapping[label_name]])  # ✅ Store label as an integer

# Create DataFrame
df = pd.DataFrame(data, columns=["directory", "image_path", "label"])
df


In [ ]:
class XrayImageDataset(Dataset):
    def __init__(self, df_imagepaths, transform=None):
        self.df_imagepaths = df_imagepaths  # ✅ Store DataFrame
        self.transform = transform  # ✅ Store transforms

    def __getitem__(self, idx):
        # ✅ Correct way to get image path
        img_path = self.df_imagepaths.iloc[idx]["image_path"]
        image = Image.open(img_path).convert("L")  # ✅ Load as grayscale

        # ✅ Correct way to get label (Already Integer!)
        label = self.df_imagepaths.iloc[idx]["label"]

        # ✅ Apply transformations if provided
        if self.transform:
            image = self.transform(image)

        return image, label  # ✅ Returns both image and numeric label

    def __len__(self):
        return len(self.df_imagepaths)  # ✅ Correctly returns dataset size


In [ ]:
xray_transform = transforms.Compose([
    transforms.Resize((64,64)),  # ✅ Resize for consistency
    transforms.ToTensor(),  # ✅ Convert to tensor
    transforms.Normalize(mean=[0.5], std=[0.5])  # ✅ Normalize for DCGAN
])

xray_dataset = XrayImageDataset(df, transform=xray_transform)
print(len(xray_dataset))  # Check total dataset size
image, label = xray_dataset[0]  # Get the first image-label pair
plt.imshow(image.squeeze(), cmap="gray")  # Display the image
print(f"Label: {label} (0: NORMAL, 1: PNEUMONIA)")  # ✅ Explicitly state what the label means


# Model Definition

In [ ]:
# ✅ Hyperparameters for ACGAN (64x64 Images)

# Random seed for reproducibility
manualSeed = 999
torch.manual_seed(manualSeed)

# Model architecture
image_size = 64  # ✅ Set for 64x64 images
nc = 1  # ✅ Number of channels (Grayscale X-ray)
nz = 100  # ✅ Latent space dimension
num_classes = 2  # ✅ Pneumonia vs Normal classification

# ✅ Reduced feature maps for smaller images
ngf = 128  # ✅ Generator feature maps (was 256, now 128)
ndf = 64   # ✅ Discriminator feature maps (was 128, now 64)
ngpu = 1   # ✅ Number of GPUs

# Training details
batch_size = 64  # ✅ Increased batch size (fits better for 64x64)
num_epochs = 5000  # ✅ Maximum epochs

# Learning rates
lr_g = 0.0002  # ✅ Generator learning rate (kept same)
lr_d = 0.00001  # ✅ ✅ Lowered Discriminator learning rate (4x slower than G)

# Optimizer settings
optimizer_type = "Adam"  # ✅ Using Adam for stability
betas = (0.5, 0.999)  # ✅ Recommended betas for GAN training

# Regularization
netd_dropout = 0.2  # ✅ ✅ Reduced dropout in Discriminator (weaken D slightly)

# Learning Rate Scheduler
scheduler_step_size = 10000  # ✅ Step size before reducing LR
scheduler_gamma = 0.5  # ✅ Decay factor

# Label smoothing settings (for stability)
real_label_smooth = 0.98  # ✅ Adjusted for real images
fake_label_smooth = 0.0  # ✅ Adjusted for fake images

# Generator update frequency (controls how often G is updated per D step)
gen_update_steps = 2  # ✅ Changed from 5 to 2 (adjustable)

# Discriminator Gaussian noise (weakens D)
discriminator_noise_std = 0.15  # ✅ Adds noise to weaken D slightly

# ✅ ACGAN-specific parameters
lambda_class = 1.0  # ✅ Weight for class prediction loss (tune if needed)
criterion_class = nn.CrossEntropyLoss()  # ✅ ACGAN: Class prediction loss

# Early stopping parameters
min_epochs = 500  # ✅ Minimum epochs before stopping
patience = 50  # ✅ Consecutive epochs without improvement before stopping
g_d_loss_delta = 0.002  # ✅ Improvement threshold
min_loss = float('inf')  # ✅ Track best loss
epochs_no_improve = 0  # ✅ Counter for early stopping
early_stop = False  # ✅ Early stopping flag


In [ ]:
# ✅ Custom weights initialization for netG and netD (ACGAN Style)
def weights_init(m):
    classname = m.__class__.__name__
    
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02) 
    
    elif isinstance(m, nn.BatchNorm2d):  
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    
    # ✅ Initialize fully connected layers in Discriminator (real/fake & class)
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)  # ✅ Xavier init for stability
        nn.init.constant_(m.bias, 0)  # ✅ Zero bias


In [ ]:
class Generator(nn.Module):
    def __init__(self, ngpu, num_classes):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.num_classes = num_classes

        # 🔹 Embed class labels into a space of the same dimension as `nz`
        self.label_emb = nn.Embedding(num_classes, nz)  # Maps labels to latent space dimension

        # 🔹 Updated Initial Transpose Layer to accept (nz + num_classes)
        self.init_layer = nn.Sequential(
            nn.ConvTranspose2d(nz + nz, ngf * 8, 4, 1, 0, bias=False),  # ✅ Expecting (nz + nz) channels now
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
        )

        # 🔹 Upsampling Blocks (3 instead of 4 for 64x64)
        self.block1 = self._make_block(ngf * 8, ngf * 4)  # 4x4 → 8x8
        self.block2 = self._make_block(ngf * 4, ngf * 2)  # 8x8 → 16x16
        self.block3 = self._make_block(ngf * 2, ngf)      # 16x16 → 32x32

        # 🔹 Final layer for 64x64 output
        self.final = nn.Sequential(
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),  # ✅ Outputs 64x64 now
            nn.Tanh()
        )

    def _make_block(self, in_channels, out_channels):
        """Helper function to create an upsampling block (NO residual skips)."""
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )

    def forward(self, noise, labels):
        """
        Forward pass for ACGAN Generator.
        Args:
        - noise: Latent noise vector (batch_size, nz)
        - labels: Class labels (batch_size)
        """
        label_embedding = self.label_emb(labels).unsqueeze(2).unsqueeze(3)  # ✅ Add dimensions for concatenation
        gen_input = torch.cat((noise, label_embedding), dim=1)  # ✅ Now has correct shape


        gen_input = gen_input.view(gen_input.shape[0], nz + nz, 1, 1)  # ✅ Adjusted for input to ConvTranspose2d
        return self.final(self.block3(self.block2(self.block1(self.init_layer(gen_input)))))


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, ngpu, num_classes):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu

        self.conv_layers = nn.Sequential(
            # Input: (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),  # (ndf) x 32 x 32
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(netd_dropout),

            # (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),  # (ndf*2) x 16 x 16
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(netd_dropout),

            # (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),  # (ndf*4) x 8 x 8
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(netd_dropout),

            # (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),  # (ndf*8) x 4 x 4
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(netd_dropout),
        )

        # 🔹 Real/Fake Output Head
        self.validity_head = nn.Sequential(
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),  # ✅ Now takes (ndf*8) input
            nn.Sigmoid()  # Output: Real/Fake probability
        )

        # 🔹 Class Prediction Output Head
        self.class_head = nn.Sequential(
            nn.Conv2d(ndf * 8, num_classes, 4, 1, 0, bias=False),  # ✅ Predicts class (num_classes)
            nn.Flatten()  
        )



    def forward(self, input):
        x = self.conv_layers(input)
        validity = self.validity_head(x).view(-1)  # ✅ Real/Fake (Binary)
        class_output = self.class_head(x)  # ✅ Class Prediction (Multiclass)
        return validity, class_output


In [ ]:
# ✅ Create the generator (ACGAN: Includes class conditioning)
netG = Generator(ngpu, num_classes).to(device)

# ✅ Explicitly reinitialize weights before training
netG.apply(weights_init)

# Handle multi-GPU if needed
if (device.type == 'cuda') and (ngpu > 1):
    netG = nn.DataParallel(netG, list(range(ngpu)))

# Print model structure AFTER moving to GPU
print("✅ Generator Model:")
print(netG)

# ✅ Create the discriminator (ACGAN: Predicts both real/fake and class labels)
netD = Discriminator(ngpu, num_classes).to(device)

# ✅ Explicitly reinitialize weights before training
netD.apply(weights_init)

# Handle multi-GPU if needed
if (device.type == 'cuda') and (ngpu > 1):
    netD = nn.DataParallel(netD, list(range(ngpu)))

# Print model structure AFTER moving to GPU
print("✅ Discriminator Model:")
print(netD)


In [ ]:
# ✅ Initialize Loss Functions for ACGAN
criterion_GAN = nn.BCELoss()  # Binary cross-entropy for real/fake classification
criterion_class = nn.CrossEntropyLoss()  # ✅ ACGAN: Class prediction loss

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move models to GPU BEFORE defining optimizers
netG.to(device)
netD.to(device)
criterion_GAN.to(device)
criterion_class.to(device)  # ✅ Move class loss to GPU

# Create batch of latent vectors for visualization
fixed_noise = torch.randn(batch_size, nz, 1, 1, device=device)
fixed_labels = torch.randint(0, num_classes, (batch_size,), dtype=torch.long, device=device)  # ✅ Explicitly set dtype to long

# Establish real and fake labels with smoothing
real_label_base = 0.9
fake_label_base = 0.1

# ✅ Use Adam for both Generator & Discriminator (ACGAN)
optimizerD = optim.Adam(netD.parameters(), lr=lr_d, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr_g, betas=(0.5, 0.999))

# Initialize learning rate schedulers for gradual decay
schedulerD = torch.optim.lr_scheduler.StepLR(optimizerD, step_size=scheduler_step_size, gamma=scheduler_gamma)
schedulerG = torch.optim.lr_scheduler.StepLR(optimizerG, step_size=scheduler_step_size, gamma=scheduler_gamma)

# Optimize DataLoader for GPU efficiency
dataloader = DataLoader(
    xray_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=lambda batch: (torch.stack([b[0] for b in batch]), torch.tensor([b[1] for b in batch], dtype=torch.long))  # ✅ Ensures labels are `long`
)


# ✅ Check if models are on GPU
print(f"Generator on GPU? {next(netG.parameters()).is_cuda}")
print(f"Discriminator on GPU? {next(netD.parameters()).is_cuda}")


# Model Training Loop

In [ ]:
import time

def display_generated_images(fake_images, epoch):
    """
    Displays a grid of generated images every n epochs.
    """
    if (epoch + 1) % 50 == 0:
        fake_images = fake_images.cpu().detach()  # ✅ Ensure tensor is detached before moving to CPU
        grid = vutils.make_grid(fake_images, padding=2, normalize=True)
        plt.figure(figsize=(8, 8))
        plt.axis("off")
        plt.title(f"Generated Images - Epoch {epoch+1}")
        plt.imshow(grid.permute(1, 2, 0))
        plt.show()

def print_processing_status(epoch, num_epochs, start_time):
    """
    Prints the training progress every n epochs with time taken.
    """
    if (epoch + 1) % 25 == 0:
        elapsed_time = time.time() - start_time
        print(f"🕒 Epoch {epoch+1}/{num_epochs} completed in {elapsed_time:.2f} sec.")


In [ ]:
# ✅ Training Loop

# Lists to track progress
img_list = []
G_losses = []
D_losses = []
iters = 0

print("🚀 Starting Training Loop...")

for epoch in range(num_epochs):
    start_time = time.time()  # Track time for each epoch

    for i, data in enumerate(dataloader, 0):

        ############################
        # (1) Update Discriminator
        ############################
        netD.zero_grad()
        real_cpu = data[0].to(device)  # ✅ Move image tensor to GPU
        real_labels = data[1].to(device, dtype=torch.long)  # ✅ Move label tensor to GPU and ensure dtype is long
        b_size = real_cpu.size(0)

        # ✅ Add Gaussian Noise to weaken D
        real_cpu += torch.randn_like(real_cpu) * discriminator_noise_std

        output_real, class_real = netD(real_cpu)  # ✅ Get both real/fake & class predictions
        
        label_real = torch.full((b_size,), 1.0, dtype=torch.float, device=device)  # ✅ Real labels = 1
        errD_real = criterion_GAN(output_real, label_real)  # ✅ Real/Fake loss
        errD_class = criterion_class(class_real.squeeze(), real_labels) # ✅ Class prediction loss
        errD_total = errD_real + lambda_class * errD_class  # ✅ Weighted sum

        errD_total.backward()
        D_x = output_real.mean().item()

        # ✅ Generate Fake Images
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_labels = torch.randint(0, num_classes, (b_size,), device=device)
        fake = netG(noise, fake_labels)

        # ✅ Add noise to fake images
        fake += torch.randn_like(fake) * discriminator_noise_std

        output_fake, class_fake = netD(fake.detach())  # ✅ Get both outputs

        label_fake = torch.full((b_size,), 0.0, dtype=torch.float, device=device)  # ✅ Fake labels = 0
        errD_fake = criterion_GAN(output_fake, label_fake)  # ✅ Fake image loss
        errD_class_fake = criterion_class(class_fake.squeeze(), fake_labels)  # ✅ Fake class prediction loss
        errD_total_fake = errD_fake + lambda_class * errD_class_fake

        errD_total_fake.backward()
        D_G_z1 = output_fake.mean().item()

        errD = errD_total + errD_total_fake
        optimizerD.step()

        ############################
        # (2) Update Generator
        ############################
        for _ in range(gen_update_steps):
            netG.zero_grad()
            fake = netG(noise, fake_labels)
            output, class_output = netD(fake)

            label_g = torch.full((b_size,), 1.0, dtype=torch.float, device=device)  # ✅ Trick D to think fake is real
            errG = criterion_GAN(output, label_g)  # ✅ Generator Adversarial Loss
            errG_class = criterion_class(class_output, fake_labels.view(-1))  # ✅ Generator Class Consistency Loss
            errG_total = errG + lambda_class * errG_class  # ✅ Weighted sum

            errG_total.backward()
            optimizerG.step()

            D_G_z2 = output.mean().item()

        # ✅ Print every 1000 iterations
        if iters % 1000 == 0:
            print(f"[{epoch}/{num_epochs}][{i}/{len(dataloader)}] "
                  f"Loss_D: {errD.item():.4f} Loss_G: {errG_total.item():.4f} "
                  f"D(x): {D_x:.4f} D(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f}")

        G_losses.append(errG_total.item())
        D_losses.append(errD.item())

        iters += 1

    # 📸 **Display generated images every 50 epochs**
    with torch.no_grad():
        fake_images = netG(fixed_noise, fixed_labels)
    display_generated_images(fake_images, epoch)

    # Save images to list for further evaluation
    img_list.append(fake_images)

    # 📉 Early stopping
    total_loss = errG_total.item() + errD.item()
    if epoch >= min_epochs:
        if min_loss - total_loss > g_d_loss_delta:
            min_loss = total_loss
            epochs_no_improve = 0
            print(f"✅ Epoch {epoch+1}: Loss improved to {min_loss:.4f}, resetting patience counter.")
        else:
            epochs_no_improve += 1
            print(f"⚠️ Epoch {epoch+1}: No significant improvement. Patience: {epochs_no_improve}/{patience}")

        if epochs_no_improve >= patience:
            print(f"⏹️ Early stopping triggered at epoch {epoch+1}. Saving final model...")
            torch.save(netG.state_dict(), "final_generator.pth")
            torch.save(netD.state_dict(), "final_discriminator.pth")
            break

    # 📉 Update learning rate scheduler
    schedulerD.step()
    schedulerG.step()

    # ✅ Print time taken for the epoch
    print_processing_status(epoch, num_epochs, start_time)

# 💾 Final model save
print("✅ Training complete. Saving final model...")
torch.save(netG.state_dict(), "final_generator.pth")
torch.save(netD.state_dict(), "final_discriminator.pth")


# Model Visualization

In [ ]:
def moving_average(values, window=50):
    return np.convolve(values, np.ones(window)/window, mode='valid')

# ✅ Dynamically set x-axis based on actual training iterations
actual_iterations = len(G_losses)  # Number of actual training steps

plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")

# Original Loss
plt.plot(range(actual_iterations), G_losses, alpha=0.3, label="G (raw)")
plt.plot(range(actual_iterations), D_losses, alpha=0.3, label="D (raw)")

# Optional: Track Classification Loss for Discriminator (if logged separately)
if 'D_class_losses' in globals():
    plt.plot(range(actual_iterations), D_class_losses, alpha=0.3, label="D (class loss)", linestyle="dotted")

# Smoothed Loss
plt.plot(range(len(moving_average(G_losses))), moving_average(G_losses), label="G (smoothed)")
plt.plot(range(len(moving_average(D_losses))), moving_average(D_losses), label="D (smoothed)")

# Optional: Smooth Class Loss (if available)
if 'D_class_losses' in globals():
    plt.plot(range(len(moving_average(D_class_losses))), moving_average(D_class_losses), label="D (class loss smoothed)", linestyle="dotted")

plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()

# ✅ Fix: Scale vertical lines to match actual training duration
for i in range(0, actual_iterations, actual_iterations // 10 if actual_iterations > 10 else 1):
    plt.axvline(x=i, color='gray', linestyle='--', alpha=0.5)

plt.show()


In [ ]:
# Grab a batch of real images from the dataloader
real_batch = next(iter(dataloader))

# Plot the real images
plt.figure(figsize=(15, 15))
plt.subplot(1, 2, 1)
plt.axis("off")
plt.title("Real Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=5, normalize=True).cpu(), (1, 2, 0)))

# ✅ Generate fake images conditioned on both classes
with torch.no_grad():
    class_0_labels = torch.zeros(32, dtype=torch.long, device=device)  # Normal X-ray class
    class_1_labels = torch.ones(32, dtype=torch.long, device=device)  # Pneumonia X-ray class

    noise = torch.randn(32, nz, 1, 1, device=device)

    fake_class_0 = netG(noise, class_0_labels)  # Generate class 0 (Normal)
    fake_class_1 = netG(noise, class_1_labels)  # Generate class 1 (Pneumonia)

# ✅ Create a visualization grid for class-conditioned fake images
fake_grid = torch.cat([fake_class_0, fake_class_1], dim=0)
fake_images = np.transpose(vutils.make_grid(fake_grid, padding=5, normalize=True).cpu(), (1, 2, 0))

# Plot the fake images from the last epoch
plt.subplot(1, 2, 2)
plt.axis("off")
plt.title("Fake Images (Top: Normal | Bottom: Pneumonia)")
plt.imshow(fake_images)
plt.show()


# Generate Synethetic Dataset

In [ ]:
import os

# Directories to save synthetic datasets
save_dir_normal = "synthetic_normal"
save_dir_pneumonia = "synthetic_pneumonia"

os.makedirs(save_dir_normal, exist_ok=True)
os.makedirs(save_dir_pneumonia, exist_ok=True)

# Number of images to generate per class
num_images_per_class = 1000  # Adjust based on need

for i in range(num_images_per_class):
    noise = torch.randn(1, nz, 1, 1, device=device)  # Generate random noise

    with torch.no_grad():
        # ✅ Generate Normal X-ray (class 0)
        fake_normal = netG(noise, torch.tensor([0], dtype=torch.long, device=device)).detach().cpu()
        # ✅ Generate Pneumonia X-ray (class 1)
        fake_pneumonia = netG(noise, torch.tensor([1], dtype=torch.long, device=device)).detach().cpu()

    # Convert tensors to PIL images and save
    normal_pil = transforms.ToPILImage()(fake_normal.squeeze(0))
    pneumonia_pil = transforms.ToPILImage()(fake_pneumonia.squeeze(0))

    normal_pil.save(f"{save_dir_normal}/normal_{i}.png")
    pneumonia_pil.save(f"{save_dir_pneumonia}/pneumonia_{i}.png")

print(f"✅ Generated {num_images_per_class} synthetic normal images in {save_dir_normal}")
print(f"✅ Generated {num_images_per_class} synthetic pneumonia images in {save_dir_pneumonia}")
